In [1]:
import sys
import os

if not os.path.exists("config.py"):
    os.chdir("backend") if os.path.exists("backend") else os.chdir("..")

sys.path.insert(0, os.getcwd())

print("Working directory:", os.getcwd())
print("config.py exists:", os.path.exists("config.py"))

Working directory: c:\Users\Dell\Documents\repo\llms\document-assistant\backend
config.py exists: True


In [2]:
from openai import OpenAI
from supabase import create_client
from config import settings

openai_client = OpenAI(api_key=settings.openai_api_key)
sb = create_client(settings.supabase_url, settings.supabase_service_role_key)

print("Clients ready")

Clients ready


In [3]:
import uuid

def create_session(title: str, project: str = None) -> dict:
    session = {
        "id": str(uuid.uuid4()),
        "title": title,
        "project": project,
    }
    result = sb.table("chat_sessions").insert(session).execute()
    return result.data[0]

session = create_session("Test Conversation — Penalties")
print("Session created:", session["id"])
print("Title:", session["title"])

Session created: dd8b305f-94e4-4682-9e96-a0bd4d6f5688
Title: Test Conversation — Penalties


In [4]:
def save_message(session_id: str, role: str, content: str, sources: list = None) -> dict:
    message = {
        "session_id": session_id,
        "role": role,
        "content": content,
        "sources": sources,
    }
    result = sb.table("chat_messages").insert(message).execute()
    return result.data[0]

# Test saving a user message
msg = save_message(session["id"], "user", "What are the penalties for late completion?")
print("Message saved:", msg["id"])

Message saved: 20b5b8c5-cb66-4a65-b832-f523d2e619b5


In [5]:
def get_messages(session_id: str) -> list[dict]:
    result = sb.table("chat_messages") \
        .select("*") \
        .eq("session_id", session_id) \
        .order("created_at") \
        .execute()
    return result.data

messages = get_messages(session["id"])
print(f"Messages in session: {len(messages)}")
for m in messages:
    print(f"  [{m['role']}]: {m['content'][:80]}")

Messages in session: 1
  [user]: What are the penalties for late completion?


In [6]:
from services.search import search_chunks
from services.prompt_builder import build_prompt

def ask_in_session(query: str, session_id: str, project: str = None, top_k: int = 5) -> dict:
    # Get last 6 messages for context
    all_messages = get_messages(session_id)
    recent_messages = all_messages[-6:] if len(all_messages) > 6 else all_messages

    # Retrieve relevant chunks
    chunks = search_chunks(query, top_k=top_k, project=project)
    context = build_prompt(query, chunks)

    # Build message history for GPT
    gpt_messages = [{"role": "system", "content": context}]
    for m in recent_messages:
        gpt_messages.append({"role": m["role"], "content": m["content"]})
    gpt_messages.append({"role": "user", "content": query})

    # Call GPT
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0.2,
        messages=gpt_messages
    )

    answer = response.choices[0].message.content
    sources = [
        {
            "clause_ref": c["clause_ref"],
            "filename": c["filename"],
            "project": c["project"],
            "similarity": round(c["similarity"], 4),
        }
        for c in chunks
    ]

    # Save both messages to Supabase
    save_message(session_id, "user", query)
    save_message(session_id, "assistant", answer, sources=sources)

    return {"answer": answer, "sources": sources}

# Test first message
result = ask_in_session(
    "What are the penalties for late completion?",
    session["id"]
)
print(result["answer"])
print("\n--- Sources ---")
for s in result["sources"]:
    print(f"  Clause {s['clause_ref']}")

The penalties for late completion are specified in Clause 17.4.1. If the Contractor fails to complete the Works, a Section, or a Key Stage within the applicable Time for Completion, the Contractor shall pay Penalties at the rate stated in section 7-A [Penalties] of Appendix 2 [Contract Particulars] for each Day or part Day that elapses beyond the Time for Completion. This applies as follows:

(a) For a failure to complete a Section, between the Time for Completion for that Section and the Taking-Over Date for that Section.
(b) For a failure to complete the Works, between the Time for Completion for the Works and the Completion Date.
(c) For a failure to achieve a Key Stage, between the Time for Completion for that Key Stage and the date on which the Engineer gives its non-objection to the completion requirements for that Key Stage as stated in the Project Brief. 

(Refer to Clause 17.4.1)

--- Sources ---
  Clause 17.4.1
  Clause 17.6.1
  Clause 3.20.4
  Clause 17.5.1
  Clause 17.7.1


In [7]:
result2 = ask_in_session(
    "Is there a maximum cap on these penalties?",
    session["id"]
)
print(result2["answer"])
print("\n--- Sources ---")
for s in result2["sources"]:
    print(f"  Clause {s['clause_ref']}")

Yes, there is a maximum cap on these penalties. According to Clause 17.3.1, the Contractor's aggregate liability to the Authority for Penalties under the Contract, other than KPI Penalties and certain health and safety or environmental penalties, shall not exceed ten per cent (10%) of the Contract Price.

--- Sources ---
  Clause 1.1.95
  Clause 17.7.1
  Clause 17.3.2
  Clause 17.3.1
  Clause 17.3.3
